<a href="https://colab.research.google.com/github/ZainabSultan2005/MLmodels/blob/main/Copy_of_PAI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [124]:
import requests
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [125]:
from google.colab import userdata

API_TOKEN = userdata.get('TMDB_TOKEN')

# Connecting to TMDB
We imported the requests library that help our notebook connect to the website. We loaded the API token in 'secret' and loaded it in our code so that we do not have to type the API token in our code. If we do that and public the notebook, unauthorized persons can use our API to wrongfully access the TMDB website.

In [126]:
url = "https://api.themoviedb.org/3/discover/movie"

In [127]:
headers = {
    "Authorization": f"Bearer {API_TOKEN}",  #TMDB recognized this pattern so we use bearer before API Token
    "accept": "application/json" #it tells TMDB that program prefers response in json format
}

response = requests.get(url, headers=headers)

print(response.status_code)

200


In [128]:
data = response.json() #converts json format to python data structures

print(data.keys())
# Testing data format
print(data["results"][0]["title"])

dict_keys(['page', 'results', 'total_pages', 'total_results'])
Iron Man


Every API request needs two parameters. First is the url and the other is authorization i.e API token. We sent the request using request libarary and get a json in response that we convert to a python dictionary. Our status code is 200 which shows that we successfully got the access.

In [129]:
all_movies = []

params = {
    "primary_release_date.gte": "2016-01-01",
    "primary_release_date.lte": "2016-12-31",
    "sort_by": "vote_count.desc",
    "page": 1
}

for i in range(5):

    params["page"] = i + 1

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    data = response.json()
    movies = data["results"]

    all_movies.extend(movies)

print(response.status_code)
print("Movies retrieved:", len(all_movies))
print(all_movies[0]["title"])
print(all_movies[-1]["title"])

200
Movies retrieved: 100
Deadpool
The Invitation


We created an empty list and defined the parameters for our data (like the release dates). Then we retrieved the relevant data, one page at a time. In total we got 100 movie. The code below ensures that in our list of movies, all movie ids are unique.

In [130]:
#Removing duplicates
def remove_duplicates(year):
    movie_ids = []
    z = 0

    for movies in year:
        id = year[z]["id"]
        movie_ids.append(id)
        z += 1

    movie_unique = set(movie_ids)

    print(len(movie_ids))
    print(len(movie_unique))


for keys, values in movies_by_year.items():
    print("Year:", keys)
    remove_duplicates(values)


Year: 2016
100
100
Year: 2017
100
100
Year: 2018
100
100
Year: 2019
100
100
Year: 2020
100
100
Year: 2021
100
100
Year: 2022
100
100
Year: 2023
100
100
Year: 2024
100
100
Year: 2025
100
100
Year: 2026
100
100


In [131]:
# Preserve the 100 movies we already collected for 2016
movies_by_year = {
    2016: all_movies.copy()
}

# Collect movies for 2017 to 2026
for year in range(2017, 2027):

    year_movies = []

    for page in range(1, 6):

        params = {
            "primary_release_date.gte": f"{year}-01-01",
            "primary_release_date.lte": f"{year}-12-31",
            "sort_by": "vote_count.desc",
            "page": page
        }

        response = requests.get(
            url,
            headers=headers,
            params=params
        )

        data = response.json()

        year_movies.extend(data["results"])

    movies_by_year[year] = year_movies

    print(year, ":", len(year_movies), "movies collected")

2017 : 100 movies collected
2018 : 100 movies collected
2019 : 100 movies collected
2020 : 100 movies collected
2021 : 100 movies collected
2022 : 100 movies collected
2023 : 100 movies collected
2024 : 100 movies collected
2025 : 100 movies collected
2026 : 100 movies collected


In [132]:
for keys,values in movies_by_year.items():
  remove_duplicates(movies_by_year[keys])


100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100


In [133]:
details_url = "https://api.themoviedb.org/3/movie"

all_movie_details = []
movie_ids = []

for keys,values in movies_by_year.items():
  for items in values:
    movie_ids.append(items["id"])



for items in movie_ids:
    response = requests.get(
        f"{details_url}/{items}",
        headers=headers
    )

    movie_details = response.json()
    all_movie_details.append(movie_details)


In [134]:
print(len(all_movie_details))
print(all_movie_details[0])



1100
{'adult': False, 'backdrop_path': '/rFj9IKlL75B2pXhZA60jkNWvxeW.jpg', 'belongs_to_collection': {'id': 448150, 'name': 'Deadpool Collection', 'poster_path': '/30c5jO7YEXuF8KiWXLg9m28GWDA.jpg', 'backdrop_path': '/dTq7mGyAR5eAydR532feWfjJjzm.jpg'}, 'budget': 58000000, 'genres': [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 35, 'name': 'Comedy'}], 'homepage': 'https://www.20thcenturystudios.com/movies/deadpool', 'id': 293660, 'imdb_id': 'tt1431045', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Deadpool', 'overview': 'The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.', 'popularity': 21.4464, 'poster_path': '/3E53WEZJqP6aM84D8CckXx4pIHw.jpg', 'production_c

As of now, we have retrieved 1100 movies from year 2016 to 2026 and verified that all movie ids are unique. Using those unique ids, we have created another list of ductionaries that contains the details of all 1100 movies. This is our data that we have to clean and train model on.

In [135]:
df = pd.DataFrame(all_movie_details)  #sare data ko humne DataFrame mein convert kar diya hai


In [136]:
df.to_csv("moviesRaw.csv", index = False)
#index=False tells pandas not to save the DataFrame's row index as an extra column in the csv file
#bcz by default index ka ik separate column banta hai jab hum csv mein dalte data

In [137]:
data = pd.read_csv("moviesRaw.csv")
#now we will inspect our data

In [138]:
data.shape  #total 1100 (rows) movies hain, with 27 (cols) attributes

(1100, 27)

In [139]:
data.head() #by default gives us the first five values

,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,revenue,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,/rFj9IKlL75B2pXhZA60jkNWvxeW.jpg,"{'id': 448150, 'name': 'Deadpool Collection', ...",58000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",https://www.20thcenturystudios.com/movies/dead...,293660,tt1431045,['US'],en,...,782837347,108,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Feel the love.,Deadpool,False,7.626,33671
1,False,/7FWlcZq3r6525LWOcvO9kNWurN1.jpg,"{'id': 131295, 'name': 'Captain America Collec...",250000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",https://www.marvel.com/movies/captain-america-...,271110,tt3498820,['US'],en,...,1155046416,147,False,"[{'english_name': 'Romanian', 'iso_639_1': 'ro...",Released,United we stand. Divided we fall.,Captain America: Civil War,False,7.456,24906
2,False,/kkoiH8ZWxJ9WSAjOadGtuHUQxbm.jpg,"{'id': 618529, 'name': 'Doctor Strange Collect...",180000000,"[{'id': 14, 'name': 'Fantasy'}, {'id': 12, 'na...",https://www.marvel.com/movies/doctor-strange,284052,tt1211837,['US'],en,...,676343174,115,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,The impossibilities are endless.,Doctor Strange,False,7.424,24249
3,False,/wAk0yKrhAmvsoMvlKs4QImhvK5X.jpg,"{'id': 531242, 'name': 'Suicide Squad Collecti...",175000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.suicidesquad.com/,297761,tt1386697,['US'],en,...,749200054,122,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Worst. Heroes. Ever.,Suicide Squad,False,5.916,22655
4,False,/8MUZz7oPXQftFTslZpRP3CVMOoq.jpg,NaN,47000000,"[{'id': 18, 'name': 'Drama'}, {'id': 878, 'nam...",http://www.arrivalmovie.com/,329865,tt2543164,['US'],en,...,203388186,116,False,"[{'english_name': 'Mandarin', 'iso_639_1': 'zh...",Released,Why are they here?,Arrival,False,7.638,20231


In [140]:
data.tail() #last five rows

,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,revenue,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count
1095,False,/4A3r1AjJg9Wa6ms3MGOScetCY0o.jpg,NaN,180000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",https://www.diggermovie.com,1248832,tt31450459,['US'],en,...,50850116,129,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,A man. A plan. A meltdown.,Digger,False,7.500,538
1096,False,/caBIySpwuFi2i7ynvHIlnxJLOdN.jpg,"{'id': 1560779, 'name': 'Batman: Knightfall Co...",0,"[{'id': 16, 'name': 'Animation'}, {'id': 28, '...",https://www.warnerbros.com/movies/batman-knigh...,1560520,tt32333324,['US'],en,...,0,79,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,NaN,Batman: Knightfall Part 1: Knightfall,False,9.093,512
1097,False,/hL5EIk48251OXlbIbRJq3yaBHk8.jpg,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 80, 'nam...",https://www.netflix.com/title/81715790,949838,tt36304003,['US'],en,...,0,105,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,The night is young. He isn't.,72 HOURS,False,6.718,512
1098,False,/uNToXatdunyvWXyXMrTI1nLvh6r.jpg,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 878, 'na...",https://www.20thcenturystudios.com/movies/mike...,1115544,tt27552099,['US'],en,...,0,107,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,"A time-traveling, double-crossing, ass-kicking...",Mike & Nick & Nick & Alice,False,6.666,508
1099,False,/seuG9UslSFybML8C4enbrE6Hsyz.jpg,NaN,3000000,"[{'id': 27, 'name': 'Horror'}, {'id': 9648, 'n...",https://ironlung.com,1116201,tt27564844,['US'],en,...,51226173,125,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,This is not an expedition. It's an execution.,Iron Lung,False,6.800,492


In [141]:
data.columns.tolist() #list of attributes agaye

['adult',
 'backdrop_path',
 'belongs_to_collection',
 'budget',
 'genres',
 'homepage',
 'id',
 'imdb_id',
 'origin_country',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'softcore',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'video',
 'vote_average',
 'vote_count']

In [142]:
data.info() #info sey hamare pass data sey related kaafi cheeze ati hai
# aur usko dekh ke humme pata chal raha hai ke hamare data mein missing / null values hain

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  1100 non-null   bool   
 1   backdrop_path          1099 non-null   object 
 2   belongs_to_collection  392 non-null    object 
 3   budget                 1100 non-null   int64  
 4   genres                 1100 non-null   object 
 5   homepage               1024 non-null   object 
 6   id                     1100 non-null   int64  
 7   imdb_id                1099 non-null   object 
 8   origin_country         1100 non-null   object 
 9   original_language      1100 non-null   object 
 10  original_title         1100 non-null   object 
 11  overview               1100 non-null   object 
 12  popularity             1100 non-null   float64
 13  poster_path            1099 non-null   object 
 14  production_companies   1100 non-null   object 
 15  prod

In [143]:
data.isna().sum() #ab woh kitni null values hain hum check kare gey isna().sum() sey

,0
adult,0
backdrop_path,1
belongs_to_collection,708
budget,0
genres,0
homepage,76
id,0
imdb_id,1
origin_country,0
original_language,0


In [144]:
data.describe()
# yahan sey humme ye pata chalta hai ke kuch values zero hain
# for example min for budget is zero, ye bhi missing values mein count hoga (bcz data is not complete)
# also agar hum data.isna().sum() kare gey tou usmein zero valued values nahi ayi gi
# for this we will use (data[col] == 0).sum() (ye kaam agey kiya hai)

,budget,id,popularity,revenue,runtime,vote_average,vote_count
count,1.100000e+03,1.100000e+03,1100.000000,1.100000e+03,1100.000000,1100.000000,1100.000000
mean,6.174349e+07,6.478979e+05,13.012696,1.882757e+08,115.119091,6.943602,4637.455455
std,7.475007e+07,3.185277e+05,27.379072,3.210666e+08,20.308526,0.673362,4396.645224
min,0.000000e+00,1.456400e+04,0.029900,0.000000e+00,12.000000,3.984000,492.000000
25%,9.000000e+06,4.018530e+05,5.514600,6.151455e+06,102.000000,6.484500,1865.000000
50%,3.500000e+07,5.652270e+05,8.075050,6.083273e+07,112.000000,6.937500,3184.500000
75%,9.000000e+07,8.412598e+05,12.526100,2.139270e+08,126.000000,7.394750,5589.000000
max,6.588000e+08,1.698863e+06,499.713600,2.942661e+09,242.000000,9.093000,33671.000000


In [145]:
data.nunique()

,0
adult,1
backdrop_path,1099
belongs_to_collection,256
budget,211
genres,493
homepage,1015
id,1100
imdb_id,1099
origin_country,56
original_language,16


In [146]:
categorical_columns = [ #frequency distribution of some columns (sare columns keleye we will use data.columns.tolist())
    "original_language",
    "status",
    "video",
    "origin_country"
]

for col in categorical_columns:
    print("\nColumn:", col)
    print(df[col].value_counts())


Column: original_language
original_language
en    1021
ja      17
fr      14
es      13
ko       9
it       6
no       4
zh       3
fi       2
pl       2
de       2
da       2
pt       2
tr       1
lv       1
te       1
Name: count, dtype: int64

Column: status
status
Released    1100
Name: count, dtype: int64

Column: video
video
False    1100
Name: count, dtype: int64

Column: origin_country
origin_country
[US]                    879
[GB]                     32
[GB, US]                 25
[JP]                     19
[US, GB]                 17
[FR]                     15
[CA, US]                 14
[ES]                     10
[KR]                      9
[AU]                      7
[AU, US]                  7
[IT]                      5
[DK]                      4
[PL]                      3
[US, CA]                  3
[CN, US]                  3
[NO]                      3
[FI]                      2
[CN, HK]                  2
[MX]                      2
[CN]                      2

In [147]:
data.duplicated().sum()

np.int64(0)

In [148]:
data.memory_usage(deep = True).sum()  # 2561442 bytes

np.int64(2561379)

In [149]:
#Data Cleaning and Preprocessing
#(step 5)
#part A (missing values)

print("total missing values: ", data.isna().sum())
print("missing percentage: ", data.isna().sum() / len(data) * 100)

total missing values:  adult                      0
backdrop_path              1
belongs_to_collection    708
budget                     0
genres                     0
homepage                  76
id                         0
imdb_id                    1
origin_country             0
original_language          0
original_title             0
overview                   0
popularity                 0
poster_path                1
production_companies       0
production_countries       0
release_date               0
revenue                    0
runtime                    0
softcore                   0
spoken_languages           0
status                     0
tagline                   43
title                      0
video                      0
vote_average               0
vote_count                 0
dtype: int64
missing percentage:  adult                     0.000000
backdrop_path             0.090909
belongs_to_collection    64.363636
budget                    0.000000
genres              

In [150]:
zero_colums = ["revenue", "budget"] #sirf ye dono columns isleye liye hain kuikeh sirf unhi dono ki min value zero thi
totalZeroValues = [0, 0]

j = 0
for i in zero_colums:
  totalZeroValues[j] += (data[i] == 0).sum()
  j += 1

print(totalZeroValues)


[np.int64(201), np.int64(159)]


In [151]:
dataCopy = data.copy() #making a copy of our original data

In [152]:
for i in zero_colums:
  dataCopy.loc[dataCopy[i] == 0, i] = np.nan

# df.loc[row, col]  gives us data of that specific row and col
# here df.loc[dataCopy[i] == 0, i] means that,
# i (col) jo hai uski woh rows jahan value 0 hai usko Nan kardo
# Nan isleye kiya hai take ik hi dafa saari rows drop kar deyn

In [153]:
dataCopy.isna().sum()
# ab yahan pe revenue keleye bhi null values arahi hain
# phele nahi tha (bcz woh values zero thin, aur isna() usko include nahi karta)

,0
adult,0
backdrop_path,1
belongs_to_collection,708
budget,159
genres,0
homepage,76
id,0
imdb_id,1
origin_country,0
original_language,0


In [154]:
dataCopy.dropna(subset=["revenue"], inplace=True) #inplace True sey actual dataCopy sey remove hojaye ga

#shape tells use that we have removed 201 null revenue rows
print(dataCopy.shape)
print(data.shape)

#uska output zero aye ga, bcz ab revenue aur budget mein koi missing value nahi hai
print(dataCopy["revenue"].isna().sum())


(899, 27)
(1100, 27)
0


In [155]:
dataCopy["budget"].describe()
#now we have to deal with the missing values in budget
#we can see here that the mean is approx 73 million and median is approx 42 million
#this tells us that our data is right skewed (contain more large values)
#thats why we are choosing to replace the missing values with the median of the data


,budget
count,8.670000e+02
mean,7.303698e+07
std,7.755143e+07
min,7.000000e+00
25%,1.800000e+07
50%,4.200000e+07
75%,1.040000e+08
max,6.588000e+08


In [156]:

dataCopy["budget"] = dataCopy["budget"].fillna(dataCopy["budget"].median())

#now we dont have any missing values left
print(dataCopy["budget"].isna().sum())

0


In [157]:
#for the column "belongs_to_collection" we are replacing the Nan with a label "no collection"
#bcz those movies do not belong to any collection
#similar reason for "homepage" and "tagline"

dataCopy["belongs_to_collection"] = dataCopy["belongs_to_collection"].fillna("no collection")
print(dataCopy["belongs_to_collection"].isna().sum())

dataCopy["homepage"] = dataCopy["homepage"].fillna("no homepage provided")
dataCopy["tagline"] = dataCopy["tagline"].fillna("no tagline provided")

print(dataCopy.isna().sum().sum())  #now we dont have any missing values

0
0


In [158]:
#making a copy of the data to introduct 5% null values

fivePercentNullData = dataCopy.copy()

#random 5% data uthaya hai
fivePercent = fivePercentNullData.sample(frac = 0.05, random_state = 1)

#jo random data uthaya hai wahan pe input_feature ko Nan kar diya
input_feature = ["budget", "runtime", "original_language"]

for i in input_feature:
  fivePercentNullData.loc[fivePercent.index, i] = np.nan

print("Five percent missing values (before):\n", fivePercentNullData.isna().sum())

Five percent missing values (before):
 adult                     0
backdrop_path             0
belongs_to_collection     0
budget                   45
genres                    0
homepage                  0
id                        0
imdb_id                   0
origin_country            0
original_language        45
original_title            0
overview                  0
popularity                0
poster_path               0
production_companies      0
production_countries      0
release_date              0
revenue                   0
runtime                  45
softcore                  0
spoken_languages          0
status                    0
tagline                   0
title                     0
video                     0
vote_average              0
vote_count                0
dtype: int64


In [159]:
fivePercentNullData[["budget", "runtime"]].describe()

,budget,runtime
count,8.540000e+02,854.000000
mean,7.173368e+07,116.544496
std,7.677931e+07,20.203995
min,3.500000e+04,57.000000
25%,1.800000e+07,102.000000
50%,4.200000e+07,113.000000
75%,1.000000e+08,128.000000
max,6.588000e+08,215.000000


In [160]:
#for budget we are filling the null values with the median
fivePercentNullData["budget"] = fivePercentNullData["budget"].fillna(fivePercentNullData["budget"].median())

#for runtime we are filling the null values with mean (bcz there is less difference between mean and median of runtime
#(suggesting that it may be symmetrically skewed))
fivePercentNullData["runtime"] = fivePercentNullData["runtime"].fillna(fivePercentNullData["runtime"].mean())

#for original_language we can fill the missing values with mode, bcz woh categorical data hai
fivePercentNullData["original_language"] = fivePercentNullData["original_language"].fillna(fivePercentNullData["original_language"].mode()[0])

#mode() returns a series, isleye mode()[0] use kiya hai

#now there are no missing values
print("after: ")
print(fivePercentNullData.isna().sum().sum())


after: 
0


In [161]:
#step 5
#part B (removing duplicates)

#checking if there exists any row with all same attributes
print("fully duplicated rows: ", dataCopy.duplicated().sum())

#cheching for duplicate movie ids
print("duplicate movie ids: ", dataCopy["id"].duplicated().sum())

#no duplicates found, so moving forward :)

fully duplicated rows:  0
duplicate movie ids:  0


In [162]:
#step 5
#part C (categorical encoding)

simple_categories = ["original_language", "origin_country", "adult", "softcore", "video", "status"]

encoder = OneHotEncoder()
